In [1]:
import sys
import os
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.translate.bleu_score import corpus_bleu

# Add parent directory for imports
sys.path.append(os.path.abspath(os.path.join('..')))
from caption import caption_image_beam_search

# Load dataset metadata - using 'payload' as the variable name
with open('../dataset/caption_datasets/dataset_coco.json', 'r') as j:
    payload = json.load(j)

# Load word map
word_map_path = '../dataset/WORDMAP_coco_5_cap_per_img_5_min_word_freq.json'
with open(word_map_path, 'r') as j:
    word_map = json.load(j)
rev_word_map = {v: k for k, v in word_map.items()}

# Model configuration
model_path = '../checkpoints/BEST_checkpoint_coco_5_cap_per_img_5_min_word_freq.pth.tar'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(model_path, map_location=str(device), weights_only=False)
decoder = checkpoint['decoder'].to(device).eval()
encoder = checkpoint['encoder'].to(device).eval()

print(f"Setup complete. Device: {torch.cuda.get_device_name(0)}")

Setup complete. Device: NVIDIA GeForce RTX 5070 Ti


In [3]:
def run_full_evaluation(beam_size=3):
    """
    Perform rigorous evaluation on the 5000-image TEST split.
    Calculates BLEU-1, BLEU-2, BLEU-3, and BLEU-4 scores.
    """
    # Filter only the independent 'test' split from the Karpathy partition
    test_data = [img for img in payload['images'] if img['split'] == 'test']
    
    references = []  # Ground truth lists
    hypotheses = []  # Model predictions
    
    print(f"Starting full evaluation on TEST split ({len(test_data)} images) with k={beam_size}...")

    # Set model to evaluation mode
    decoder.eval()
    encoder.eval()

    with torch.no_grad():
        for i, img_obj in enumerate(test_data):
            fpath = os.path.join("../dataset/val2014", img_obj['filename'])
            
            # Inference using Beam Search
            seq, _ = caption_image_beam_search(encoder, decoder, fpath, word_map, beam_size)
            
            # Prepare ground truth: list of lists of tokens
            img_refs = [c['tokens'] for c in img_obj['sentences']]
            references.append(img_refs)
            
            # Prepare hypothesis: list of tokens (excluding special tags)
            prediction = [rev_word_map[ind] for ind in seq if ind not in 
                          {word_map['<start>'], word_map['<end>'], word_map['<pad>']}]
            hypotheses.append(prediction)
            
            # Status update every 500 images
            if (i + 1) % 500 == 0:
                print(f"Progress: {i + 1}/{len(test_data)} images processed.")

    # Calculate corpus-level BLEU scores with cumulative weights
    # BLEU-1: 1-gram
    # BLEU-2: 1-gram to 2-gram
    # BLEU-3: 1-gram to 3-gram
    # BLEU-4: 1-gram to 4-gram
    b1 = corpus_bleu(references, hypotheses, weights=(1.0, 0, 0, 0))
    b2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0))
    b3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0))
    b4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25))
    
    return {"BLEU-1": b1, "BLEU-2": b2, "BLEU-3": b3, "BLEU-4": b4}

# Execute evaluation
# Using k=3 as it showed a strong balance in Experiment 1
final_results = run_full_evaluation(beam_size=3)

# Display results
print("\nFinal Results Table:")
for metric, score in final_results.items():
    print(f"{metric}: {score:.4f}")

Starting full evaluation on TEST split (5000 images) with k=3...
Progress: 500/5000 images processed.
Progress: 1000/5000 images processed.
Progress: 1500/5000 images processed.
Progress: 2000/5000 images processed.
Progress: 2500/5000 images processed.
Progress: 3000/5000 images processed.
Progress: 3500/5000 images processed.
Progress: 4000/5000 images processed.
Progress: 4500/5000 images processed.
Progress: 5000/5000 images processed.

Final Results Table:
BLEU-1: 0.7365
BLEU-2: 0.5680
BLEU-3: 0.4342
BLEU-4: 0.3263
